In [1]:
!apt install fluidsynth

!git clone https://github.com/jthickstun/anticipation.git
!pip install ./anticipation
!pip install -r anticipation/requirements.txt

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  fluid-soundfont-gm libevdev2 libfluidsynth3 libgudev-1.0-0 libinput-bin
  libinput10 libinstpatch-1.0-2 libmd4c0 libmtdev1 libqt5core5a libqt5dbus5
  libqt5gui5 libqt5network5 libqt5svg5 libqt5widgets5 libwacom-bin
  libwacom-common libwacom9 libxcb-icccm4 libxcb-image0 libxcb-keysyms1
  libxcb-render-util0 libxcb-util1 libxcb-xinerama0 libxcb-xinput0 libxcb-xkb1
  libxkbcommon-x11-0 qsynth qt5-gtk-platformtheme qttranslations5-l10n
  timgm6mb-soundfont
Suggested packages:
  fluid-soundfont-gs qt5-image-formats-plugins qtwayland5 jackd
The following NEW packages will be installed:
  fluid-soundfont-gm fluidsynth libevdev2 libfluidsynth3 libgudev-1.0-0
  libinput-bin libinput10 libinstpatch-1.0-2 libmd4c0 libmtdev1 libqt5core5a
  libqt5dbus5 libqt5gui5 libqt5network5 libqt5svg5 libqt5widgets5 libwacom-bin
  libwacom-common libwacom9 libx

In [2]:
import sys,time
import numpy as np

import midi2audio
import transformers
import os

import torch
import torch.nn.functional as F

import random

from transformers import AutoModelForCausalLM
from transformers import BertConfig, BertModel
from pathlib import Path
from IPython.display import Audio

from anticipation import ops
from anticipation.sample import generate
from anticipation.tokenize import extract_instruments
from anticipation.convert import events_to_midi,midi_to_events
from anticipation.config import *
from anticipation.vocab import *


os.environ['TORCH_USE_CUDA_DSA'] = '1'
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import torch
from transformers import BertConfig, BertModel

# set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# rebuild configuration
configuration = BertConfig()
configuration.vocab_size = 55128
configuration.max_position_embeddings = 2048

# load model weights
structure_derivation_model = BertModel(configuration).to(device)

# load the saved state
checkpoint_path = "/content/drive/MyDrive/MusicData/bert_checkpoints/structure_derivation_model.pth"
structure_derivation_model.load_state_dict(torch.load(checkpoint_path, map_location=device))

# set to evaluation mode
structure_derivation_model.eval()

Using device: cuda


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(55128, 768, padding_idx=0)
    (position_embeddings): Embedding(2048, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
 

In [5]:
# function to segment a generated piece into 10-second chunks
def segment(sequence, total_duration, segment_length):
  tokens_per_second = len(sequence) / total_duration
  tokens_per_segment = int(tokens_per_second * segment_length)

  segments = []
  for i in range(0, len(sequence), tokens_per_segment):
      segment = sequence[i:i + tokens_per_segment]
      if len(segment) == tokens_per_segment:
          segments.append(segment)

  return segments

In [7]:
original_mid_1 = '/content/drive/MyDrive/MusicData/qualtrics/Beautiful_In_My_Eyes.mid'
tokenised_midi = midi_to_events(original_mid_1)
segments = segment(tokenised_midi, 60, 10)
input_ids_list = [torch.tensor(s).unsqueeze(0).to(device) for s in segments]
embeddings = []
for input_ids in input_ids_list:
    output = structure_derivation_model(input_ids=input_ids)
    emb = output.last_hidden_state[:, 0]  # CLS token
    emb = F.normalize(emb, dim=-1)        # normalise output
    embeddings.append(emb)
anchor = embeddings[0]
others = embeddings[1:]

scores = [
    F.cosine_similarity(anchor, other, dim=-1).item()
    for other in others
]
print (np.mean(scores))

0.8616712927818299
